# Zero-DCE++ with SE-Block — clean rebuild (v12: fixed early-stopping baseline)

## Why this exists
v11's curriculum was the right idea, but had a real bug: the early-stopping
baseline (`best_loss`/`no_improve`) only reset when the ramp FINISHED, not
when it started. From the moment the streams turn on, every epoch's total
loss includes extra terms (`w_normal_over * loss_normal`, `w_normal_over *
loss_over`) that are always >= 0 -- so it's mathematically impossible to
beat the pre-ramp best loss while those terms are active, no matter how
well the model is actually learning. `no_improve` was guaranteed to climb
for the entire ramp length and trip `PATIENCE` before the ramp ever
finished. This is exactly what two actual runs showed: training proceeds
fine through the dark-only phase, then stops early shortly after the
curriculum streams activate, with no real signal about whether the new
streams were converging.

**v12 fixes this with one change:** the baseline resets the epoch the
streams first turn ON (`streams_active` first becomes `True`), not when
the ramp completes. Everything else -- curriculum schedule, weights,
per-stream `backward()`, noise warm-up -- is unchanged from v11.

### 1. Imports

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import cv2
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 3060


### 2. SE-Block

Now configurable: `activation` controls the bottleneck's non-linearity
(`relu`, `prelu`, `silu`), `gate` controls the final channel-gate function
(`sigmoid`, `hardsigmoid`). Defaults (`relu`, `sigmoid`) exactly match the
v7 behavior, so nothing changes unless you deliberately pass a different
combination when constructing `DCENetLite`.

In [2]:
class SEBlock(nn.Module):
    _ACT = {
        'relu':  lambda: nn.ReLU(inplace=True),
        'prelu': lambda: nn.PReLU(),
        'silu':  lambda: nn.SiLU(inplace=True),   # aka Swish, used in EfficientNet's SE
        'gelu':  lambda: nn.GELU(),
    }
    _GATE = {
        'sigmoid':     lambda: nn.Sigmoid(),
        'hardsigmoid': lambda: nn.Hardsigmoid(inplace=True),  # used in MobileNetV3's SE
    }

    def __init__(self, channels, reduction=4, activation='relu', gate='sigmoid'):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            self._ACT[activation](),
            nn.Linear(channels // reduction, channels, bias=False),
            self._GATE[gate](),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w

print('SE-Block defined (activation/gate configurable; defaults match v7: relu/sigmoid)')

SE-Block defined (activation/gate configurable; defaults match v7: relu/sigmoid)


### 3. Depth-Separable Convolution + DCE-Net++

Unchanged from v8 except `enhance_image_with_curve`, which now takes
`max_exposure` (default `MAX_EXPOSURE = 0.95`, defined right above it).
As each curve iteration runs, only the *brightening* part of that step
(`a*(x^2-x) > 0` in this repo's sign convention) is smoothly damped as a
pixel approaches `max_exposure`; darkening is untouched. This guarantees no
pixel can exceed the ceiling regardless of what the network outputs --
including on well-lit inputs at inference, which is exactly the case this
was added for. Set `MAX_EXPOSURE` closer to 1.0 for a looser ceiling
(0.90-0.97 is a reasonable range to try).

In [3]:
class DepthSeparableConv2d(nn.Module):
    """Depthwise conv (per-channel spatial filter) + pointwise 1x1 (channel mixing).
    Matches TF's SeparableConv2D: depth_multiplier=1, RandomNormal(std=0.02) init."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding,
                                    groups=in_channels, bias=True)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=True)
        nn.init.normal_(self.depthwise.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pointwise.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.depthwise.bias)
        nn.init.zeros_(self.pointwise.bias)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class DCENetLite(nn.Module):
    """Zero-DCE++ backbone: depth-separable convs, SE-Block(s) placeable at any
    of conv1-conv4, single shared 3-channel curve map (not per-iteration 24-channel)."""
    def __init__(self, filters=32, use_attention=True, se_positions=(3,),
                 reduction=4, se_activation='relu', se_gate='sigmoid', iterations=8):
        super().__init__()
        self.use_attention = use_attention
        self.se_positions  = set(se_positions) if use_attention else set()
        self.iterations    = iterations

        self.conv1 = DepthSeparableConv2d(3,          filters)
        self.conv2 = DepthSeparableConv2d(filters,    filters)
        self.conv3 = DepthSeparableConv2d(filters,    filters)
        self.conv4 = DepthSeparableConv2d(filters,    filters)
        self.conv5 = DepthSeparableConv2d(filters*2,  filters)  # concat(x3, x4)
        self.conv6 = DepthSeparableConv2d(filters*2,  filters)  # concat(x2, x5)
        self.conv7 = DepthSeparableConv2d(filters*2,  3)        # concat(x1, x6) -> 3ch only
        self.relu  = nn.ReLU(inplace=True)

        if use_attention:
            self.se_blocks = nn.ModuleDict({
                str(pos): SEBlock(filters, reduction=reduction,
                                   activation=se_activation, gate=se_gate)
                for pos in self.se_positions
            })

    def _se(self, feat, pos):
        if pos in self.se_positions:
            return self.se_blocks[str(pos)](feat)
        return feat

    def forward(self, x):
        x1 = self._se(self.relu(self.conv1(x)),  1)
        x2 = self._se(self.relu(self.conv2(x1)), 2)
        x3 = self._se(self.relu(self.conv3(x2)), 3)
        x4 = self._se(self.relu(self.conv4(x3)), 4)
        x5 = self.relu(self.conv5(torch.cat([x3, x4], 1)))
        x6 = self.relu(self.conv6(torch.cat([x2, x5], 1)))
        a_map = torch.tanh(self.conv7(torch.cat([x1, x6], 1)))  # [B, 3, H, W]
        return a_map


MAX_EXPOSURE = 0.95  # ceiling: no pixel can be pushed brighter than this by the curve steps

def enhance_image_with_curve(x, a_map, iterations=8, max_exposure=0.95, knee=0.10):
    """Curve identical to v7/no-ceiling behavior. A soft ceiling is applied
    only at the very end, only to pixels within `knee` of full brightness --
    everything below (max_exposure - knee) passes through completely
    unchanged, so normal LOL-v2 training dynamics (which converge around
    ~0.5) are untouched. Only genuinely near-blown-out pixels get gently
    compressed toward max_exposure, which is the actual overexposure case
    this is meant to guard against."""
    enhanced = x
    for _ in range(iterations):
        enhanced = enhanced + a_map * (enhanced.pow(2) - enhanced)
    enhanced = torch.clamp(enhanced, 0.0, 1.0)

    threshold = max_exposure - knee
    over = (enhanced > threshold).float()
    t = ((enhanced - threshold) / (1.0 - threshold + 1e-8)).clamp(0, 1)
    compressed = threshold + knee * (1.0 - torch.exp(-3.0 * t))
    compressed = torch.minimum(compressed, torch.full_like(enhanced, max_exposure))

    return enhanced * (1 - over) + compressed * over


# se_positions=(3,), reduction=4, se_activation='relu', se_gate='sigmoid'
# reproduces v7's SE config exactly. Change any of these for the ablation table.
model = DCENetLite(use_attention=True, se_positions=(3,),
                    reduction=4, se_activation='relu', se_gate='sigmoid').to(device)
print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'MAX_EXPOSURE ceiling: {MAX_EXPOSURE}')

Model: 11,073 parameters
MAX_EXPOSURE ceiling: 0.95


### 4. Loss Functions (ported from tested TF reference) + noise-suppression term + adaptive exposure

Two changes from v8:

- **`ExposureControlLoss`** now takes the input image too. If an input patch
  is already at or above the target brightness `e`, the target becomes the
  input's own level (i.e. "don't push further") instead of always demanding
  `e=0.52` -- this is what teaches the network to hold back on already-bright
  input rather than only being capped after the fact by `MAX_EXPOSURE`.
- **`NoiseSuppressionLoss`** rewritten to fix the mode-collapse bug from the
  last run: it now penalizes the *increase* in local variation from input to
  enhanced (never negative, so its minimum is "no added noise", not "flat
  image"), and dark-region weighting uses a threshold instead of
  `(1 - input.mean())`. `w_noise` lowered to 5.0 and ramped in via a 20-epoch
  warm-up in the training loop below (`loss_fn.w_noise` starts at 0 and
  reaches 5.0 by epoch 20), so it has no effect until the other four terms
  have already stabilized, exactly like v7 behaved before any noise term
  existed.

In [4]:
class SpatialConsistencyLoss(nn.Module):
    def __init__(self):
        super().__init__()
        k_left  = torch.tensor([[0.,0.,0.],[-1.,1.,0.],[0.,0.,0.]]).view(1,1,3,3)
        k_right = torch.tensor([[0.,0.,0.],[0.,1.,-1.],[0.,0.,0.]]).view(1,1,3,3)
        k_up    = torch.tensor([[0.,-1.,0.],[0.,1.,0.],[0.,0.,0.]]).view(1,1,3,3)
        k_down  = torch.tensor([[0.,0.,0.],[0.,1.,0.],[0.,-1.,0.]]).view(1,1,3,3)
        self.register_buffer('k_left',  k_left)
        self.register_buffer('k_right', k_right)
        self.register_buffer('k_up',    k_up)
        self.register_buffer('k_down',  k_down)
        self.pool = nn.AvgPool2d(kernel_size=4, stride=4)

    def forward(self, inp, enh):
        inp_mean = inp.mean(dim=1, keepdim=True)
        enh_mean = enh.mean(dim=1, keepdim=True)
        inp_pool = self.pool(inp_mean)
        enh_pool = self.pool(enh_mean)

        d_inp = [F.conv2d(inp_pool, k, padding=1) for k in
                 (self.k_left, self.k_right, self.k_up, self.k_down)]
        d_enh = [F.conv2d(enh_pool, k, padding=1) for k in
                 (self.k_left, self.k_right, self.k_up, self.k_down)]

        return sum((di - de).pow(2) for di, de in zip(d_inp, d_enh)).mean()


class ExposureControlLoss(nn.Module):
    """Target brightness now adapts toward the input's own level when the
    input patch is already at/above target, instead of always demanding
    e=0.52 regardless of the input's own brightness. This is what teaches
    the network to hold back on already-bright input, complementing the
    hard MAX_EXPOSURE ceiling in enhance_image_with_curve."""
    def __init__(self, e=0.52, patch_size=16):
        super().__init__()
        self.e = e
        self.pool = nn.AvgPool2d(kernel_size=patch_size, stride=patch_size)

    def forward(self, enhanced, input_img):
        mean_c       = enhanced.mean(dim=1, keepdim=True)
        input_c      = input_img.mean(dim=1, keepdim=True)
        pooled       = self.pool(mean_c)
        input_pooled = self.pool(input_c)
        target = torch.where(input_pooled >= self.e, input_pooled,
                              torch.full_like(input_pooled, self.e))
        return (pooled - target).pow(2).mean()


class ColorConstancyLoss(nn.Module):
    def forward(self, enhanced):
        mean_rgb = enhanced.mean(dim=[2, 3], keepdim=True)  # [B, 3, 1, 1]
        mr, mg, mb = mean_rgb[:, 0:1], mean_rgb[:, 1:2], mean_rgb[:, 2:3]
        d_rg = (mr - mg).pow(2)
        d_rb = (mr - mb).pow(2)
        d_gb = (mb - mg).pow(2)
        return torch.sqrt(d_rg + d_rb + d_gb + 1e-8).mean()


class IlluminationSmoothnessLoss(nn.Module):
    def forward(self, alpha_map):
        b, c, h, w = alpha_map.shape
        count_h = (w - 1) * c
        count_w = w * (c - 1)

        tv_h = (alpha_map[:, :, 1:, :] - alpha_map[:, :, :h-1, :]).pow(2).sum() / count_h
        tv_w = (alpha_map[:, :, :, 1:] - alpha_map[:, :, :, :w-1]).pow(2).sum() / count_w

        return 2 * (tv_h + tv_w) / b


class NoiseSuppressionLoss(nn.Module):
    """Penalizes the INCREASE in local pixel variation from input to enhanced
    (not the enhanced image's raw variation), so it can't be minimized by
    flattening the whole image to black -- its minimum is reached when
    enhanced's local texture matches the input's. Dark-region weighting uses
    a threshold (dark_threshold) rather than (1 - mean), since for a
    low-light dataset nearly the whole frame is dark and (1-mean) ends up
    weighting almost everything near 1."""
    def __init__(self, dark_threshold=0.3):
        super().__init__()
        self.dark_threshold = dark_threshold

    def forward(self, enhanced, input_img):
        def grad(t):
            gh = (t[:, :, 1:, :] - t[:, :, :-1, :]).abs()
            gw = (t[:, :, :, 1:] - t[:, :, :, :-1]).abs()
            return gh, gw

        enh_gh, enh_gw = grad(enhanced)
        inp_gh, inp_gw = grad(input_img)

        excess_h = (enh_gh - inp_gh).clamp(min=0)
        excess_w = (enh_gw - inp_gw).clamp(min=0)

        input_mean  = input_img.mean(dim=1, keepdim=True)
        dark_weight = torch.clamp((self.dark_threshold - input_mean) / self.dark_threshold, 0, 1)

        return (excess_h * dark_weight[:, :, 1:, :]).mean() + (excess_w * dark_weight[:, :, :, 1:]).mean()


class DenoiseHead(nn.Module):
    """Lightweight learned denoiser applied after curve enhancement.
    Depthwise-separable, residual, output scaled small (x0.1) so it corrects
    noise rather than repainting the image."""
    def __init__(self, channels=16):
        super().__init__()
        self.conv1 = DepthSeparableConv2d(3, channels)
        self.conv2 = DepthSeparableConv2d(channels, 3)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, enhanced):
        residual = self.relu(self.conv1(enhanced))
        residual = torch.tanh(self.conv2(residual)) * 0.1
        return torch.clamp(enhanced + residual, 0, 1)


class ZeroDCELoss(nn.Module):
    """Combines all four zero-reference terms (reference repo's LITE weights,
    as tuned in v7) plus a noise-suppression term, ramped in via warm-up."""
    def __init__(self, e=0.52, patch_size=16,
                 w_spa=5.0, w_exp=20.0, w_col=5.0, w_ill=400.0, w_noise=5.0):
        super().__init__()
        self.spa = SpatialConsistencyLoss()
        self.exp = ExposureControlLoss(e=e, patch_size=patch_size)
        self.col = ColorConstancyLoss()
        self.ill = IlluminationSmoothnessLoss()
        self.w_spa, self.w_exp, self.w_col, self.w_ill, self.w_noise = \
            w_spa, w_exp, w_col, w_ill, w_noise

    def forward(self, enhanced, input_img, a_map, l_noise=None):
        l_spa = self.spa(input_img, enhanced)
        l_exp = self.exp(enhanced, input_img)
        l_col = self.col(enhanced)
        l_ill = self.ill(a_map)
        if l_noise is None:
            l_noise = torch.zeros((), device=enhanced.device)
        total = (self.w_spa * l_spa + self.w_exp * l_exp
                 + self.w_col * l_col + self.w_ill * l_ill
                 + self.w_noise * l_noise)
        return total, l_spa, l_exp, l_col, l_ill, l_noise

loss_fn       = ZeroDCELoss().to(device)
noise_loss_fn = NoiseSuppressionLoss().to(device)
denoise_head  = DenoiseHead().to(device)
print('Loss functions defined (lite weights: spa=5, exp=20, col=5, ill=400, noise=5 [ramped])')
print(f'DenoiseHead: {sum(p.numel() for p in denoise_head.parameters()):,} parameters')

Loss functions defined (lite weights: spa=5, exp=20, col=5, ill=400, noise=5 [ramped])
DenoiseHead: 305 parameters


### 5. Dataset (LOL-v2 Real_captured, paired Low/Normal)

Unchanged loading/augmentation. One behavioral note: `normal` was always
loaded in every batch (previously only used as eval ground truth) — v10's
training loop now also feeds it to the model directly as a second training
stream, plus a synthetically-overexposed third stream derived from it. No
dataset changes needed here, only how the training loop in Section 6 uses
what's already being loaded.

In [5]:
class LOLDataset(Dataset):
    def __init__(self, image_pairs, img_size=512, augment=False):
        self.image_pairs = image_pairs
        self.img_size    = img_size
        self.augment     = augment
        self.resize_size = int(img_size * 1.15)

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        pair = self.image_pairs[idx]
        low_img    = Image.open(pair['low']).convert('RGB')
        normal_img = Image.open(pair['normal']).convert('RGB')

        # Resize by SHORT SIDE — preserves aspect ratio
        low_img    = TF.resize(low_img,    self.resize_size, interpolation=Image.BILINEAR)
        normal_img = TF.resize(normal_img, self.resize_size, interpolation=Image.BILINEAR)

        if self.augment:
            i, j, h, w = transforms.RandomCrop.get_params(
                low_img, (self.img_size, self.img_size))
            low_img    = TF.crop(low_img,    i, j, h, w)
            normal_img = TF.crop(normal_img, i, j, h, w)
            if torch.rand(1) > 0.5:
                low_img    = TF.hflip(low_img)
                normal_img = TF.hflip(normal_img)
        else:
            low_img    = TF.center_crop(low_img,    self.img_size)
            normal_img = TF.center_crop(normal_img, self.img_size)

        return {
            'low':      TF.to_tensor(low_img),
            'normal':   TF.to_tensor(normal_img),
            'filename': pair['filename']
        }

def load_image_pairs(data_path):
    low_p  = data_path / 'Low'
    norm_p = data_path / 'Normal'
    lows   = sorted([f for f in os.listdir(low_p)  if f.endswith(('.jpg','.png'))])
    norms  = sorted([f for f in os.listdir(norm_p) if f.endswith(('.jpg','.png'))])
    assert len(lows)==len(norms), f'Pair mismatch: {len(lows)} vs {len(norms)}'
    return [{'low':str(low_p/l),'normal':str(norm_p/n),'filename':l}
            for l,n in zip(lows,norms)]

BASE_PATH    = Path('LOL-v2')
train_pairs  = load_image_pairs(BASE_PATH / 'Real_captured' / 'Train')
test_pairs   = load_image_pairs(BASE_PATH / 'Real_captured' / 'Test')
print(f'LOL-v2: {len(train_pairs)} train, {len(test_pairs)} test')

BATCH_SIZE  = 8
IMG_SIZE    = 512
train_loader = DataLoader(LOLDataset(train_pairs, IMG_SIZE, augment=True),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(LOLDataset(test_pairs,  IMG_SIZE, augment=False),
                          batch_size=1,          shuffle=False, num_workers=0)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

LOL-v2: 689 train, 100 test
Train batches: 87 | Test batches: 100


### 6. Training (curriculum: dark-only, then ramp in normal/over streams)

Same core loop as v10 (gradient accumulation, per-stream `backward()` to
avoid the v10 OOM, noise loss detached from the curve network, 20-epoch
noise warm-up). New: `CURRICULUM_START`/`CURRICULUM_LEN`/
`W_NORMAL_OVER_TARGET` control when and how strongly the normal/
synthetic-overexposed streams are introduced. Before `CURRICULUM_START`,
those streams aren't run at all (saves compute); after the ramp finishes,
the early-stopping baseline is reset once, since the loss scale changes
as the extra terms are added in.

In [6]:
def synthetic_overexpose(img, gain_range=(1.5, 3.0)):
    """Randomly blows out an already-normal image to create a free
    'overexposed' training example -- teaches the network to use the
    darkening half of its curve range (a>0 in this repo's sign
    convention), which it otherwise has zero incentive to ever learn."""
    gain = torch.empty(img.size(0), 1, 1, 1, device=img.device).uniform_(*gain_range)
    return torch.clamp(img * gain, 0, 1)


LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-5
NUM_EPOCHS    = 150
PATIENCE      =  35
ACCUM_STEPS   = 2
NOISE_WARMUP_EPOCHS = 25
NOISE_TARGET_WEIGHT = 5.0

# Curriculum for the normal/synthetic-overexposed streams -- see intro cell.
CURRICULUM_START    = 25
CURRICULUM_LEN      = 20
W_NORMAL_OVER_TARGET = 0.3

CHECKPOINT = 'best_zerodce_pp_seblock_v11.pth'

optimizer = optim.Adam(list(model.parameters()) + list(denoise_head.parameters()),
                        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-7)

history = {'loss':[],'spa':[],'exp':[],'col':[],'ill':[],'noise':[],
           'loss_normal':[],'loss_over':[],'w_normal_over':[],'lr':[]}
best_loss     = float('inf')
no_improve    = 0
stopped_epoch = NUM_EPOCHS
ramp_started  = False   # True the FIRST epoch w_normal_over > 0 (not when ramp finishes)

print(f"{'='*60}")
print(f'TRAINING Zero-DCE++ (depth-separable) + SE-Block + DenoiseHead + curriculum multi-exposure \u2014 max {NUM_EPOCHS} epochs, patience={PATIENCE}')
print(f'Effective batch: {BATCH_SIZE}\u00d7{ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}')
print(f'MAX_EXPOSURE ceiling: {MAX_EXPOSURE} | Noise warm-up: {NOISE_WARMUP_EPOCHS} epochs -> w_noise={NOISE_TARGET_WEIGHT}')
print(f'Curriculum: dark-only until epoch {CURRICULUM_START}, then normal/over ramp to w={W_NORMAL_OVER_TARGET} over {CURRICULUM_LEN} epochs')
print(f"{'='*60}")

for epoch in range(NUM_EPOCHS):
    model.train()
    denoise_head.train()
    totals = dict(loss=0, spa=0, exp=0, col=0, ill=0, noise=0, loss_normal=0, loss_over=0)

    loss_fn.w_noise = NOISE_TARGET_WEIGHT * min(1.0, epoch / NOISE_WARMUP_EPOCHS)

    progress       = max(0.0, min(1.0, (epoch - CURRICULUM_START) / CURRICULUM_LEN))
    w_normal_over  = W_NORMAL_OVER_TARGET * progress
    streams_active = w_normal_over > 0.0

    # Reset the early-stopping baseline the moment the streams first turn ON,
    # not when the ramp finishes. From this epoch onward, every loss value
    # includes extra, ALWAYS-POSITIVE terms that a pre-ramp score can never
    # be beaten by -- resetting here lets genuine improvement during the
    # ramp register as improvement, instead of guaranteeing no_improve climbs
    # for the entire ramp length regardless of how training is actually going
    # (this was the actual cause of early stopping triggering with no real
    # convergence check happening -- it was mathematically guaranteed to
    # trigger partway through the ramp, every run).
    if streams_active and not ramp_started:
        ramp_started = True
        best_loss  = float('inf')
        no_improve = 0
        print(f'  [Epoch {epoch+1}] curriculum streams now active -- resetting early-stop baseline')

    optimizer.zero_grad()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'[{epoch+1}/{NUM_EPOCHS}] w_no={w_normal_over:.3f}')

    for step, batch in pbar:
        low_imgs    = batch['low'].to(device)
        normal_imgs = batch['normal'].to(device)

        # --- Stream 1: genuinely dark input (teaches brightening) ---
        a_map        = model(low_imgs)
        enhanced_pre = enhance_image_with_curve(low_imgs, a_map)
        enhanced     = denoise_head(enhanced_pre)
        l_noise      = noise_loss_fn(denoise_head(enhanced_pre.detach()), low_imgs)
        loss_low, l_spa, l_exp, l_col, l_ill, l_noise = loss_fn(enhanced, low_imgs, a_map, l_noise)
        (loss_low / ACCUM_STEPS).backward()

        if streams_active:
            # --- Stream 2: already-correct input ("leave it alone") ---
            a_map_n        = model(normal_imgs)
            enhanced_n_pre = enhance_image_with_curve(normal_imgs, a_map_n)
            enhanced_n     = denoise_head(enhanced_n_pre)
            loss_normal, *_ = loss_fn(enhanced_n, normal_imgs, a_map_n)
            (w_normal_over * loss_normal / ACCUM_STEPS).backward()

            # --- Stream 3: synthetically overexposed input ("darken this") ---
            over_imgs      = synthetic_overexpose(normal_imgs)
            a_map_o        = model(over_imgs)
            enhanced_o_pre = enhance_image_with_curve(over_imgs, a_map_o)
            enhanced_o     = denoise_head(enhanced_o_pre)
            loss_over, *_  = loss_fn(enhanced_o, over_imgs, a_map_o)
            (w_normal_over * loss_over / ACCUM_STEPS).backward()
        else:
            loss_normal = torch.zeros((), device=device)
            loss_over   = torch.zeros((), device=device)

        loss = loss_low + w_normal_over * loss_normal + w_normal_over * loss_over  # logging only

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(denoise_head.parameters()), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        totals['loss']        += loss.item()
        totals['spa']         += l_spa.item()
        totals['exp']         += l_exp.item()
        totals['col']         += l_col.item()
        totals['ill']         += l_ill.item()
        totals['noise']       += l_noise.item()
        totals['loss_normal'] += loss_normal.item()
        totals['loss_over']   += loss_over.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'exp': f'{l_exp.item():.4f}',
                           'normal': f'{loss_normal.item():.4f}', 'over': f'{loss_over.item():.4f}'})

    n = len(train_loader)
    avg_loss = totals['loss'] / n
    for k in ['loss','spa','exp','col','ill','noise','loss_normal','loss_over']:
        history[k].append(totals[k] / n)
    history['w_normal_over'].append(w_normal_over)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    if (epoch + 1) % 10 == 0:
        model.eval(); denoise_head.eval()
        with torch.no_grad():
            sample = next(iter(train_loader))['low'].to(device)
            a_s    = model(sample)
            e_s    = denoise_head(enhance_image_with_curve(sample, a_s))
            mean_brightness = e_s.mean().item()
        model.train(); denoise_head.train()
        print(f'  [Epoch {epoch+1}] mean_brightness={mean_brightness:.4f}', end='')
        if mean_brightness < 0.1:
            print(' \u2014 COLLAPSE DETECTED, stopping!')
            stopped_epoch = epoch + 1
            break
        else:
            print(f' | no_improve={no_improve}/{PATIENCE}')

    if avg_loss < best_loss:
        best_loss  = avg_loss
        no_improve = 0
        torch.save({'model': model.state_dict(),
                     'denoise_head': denoise_head.state_dict()}, CHECKPOINT)
        print(f'  Epoch {epoch+1}: loss={avg_loss:.6f}  [BEST saved]')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            stopped_epoch = epoch + 1
            print(f'  Early stopping at epoch {stopped_epoch}')
            break

    scheduler.step()

print(f"\n{'='*60}")
print(f'DONE \u2014 best loss: {best_loss:.6f} | stopped at epoch: {stopped_epoch}')
print(f"{'='*60}")

TRAINING Zero-DCE++ (depth-separable) + SE-Block + DenoiseHead + curriculum multi-exposure — max 150 epochs, patience=35
Effective batch: 8×2 = 16
MAX_EXPOSURE ceiling: 0.95 | Noise warm-up: 25 epochs -> w_noise=5.0
Curriculum: dark-only until epoch 25, then normal/over ramp to w=0.3 over 20 epochs


[1/150] w_no=0.000: 100%|██████████| 87/87 [00:46<00:00,  1.87it/s, loss=4.3126, exp=0.2096, normal=0.0000, over=0.0000]


  Epoch 1: loss=4.376483  [BEST saved]


[2/150] w_no=0.000: 100%|██████████| 87/87 [00:45<00:00,  1.91it/s, loss=3.9601, exp=0.1964, normal=0.0000, over=0.0000]


  Epoch 2: loss=4.295650  [BEST saved]


...

  [Epoch 50] mean_brightness=0.2684 | no_improve=22/35


[60/150] w_no=0.300: 100%|██████████| 87/87 [01:42<00:00,  1.18s/it, loss=3.4356, exp=0.0911, normal=1.8221, over=1.8533]


  [Epoch 60] mean_brightness=0.3213 | no_improve=32/35


[62/150] w_no=0.300: 100%|██████████| 87/87 [01:44<00:00,  1.20s/it, loss=3.3651, exp=0.0340, normal=1.8148, over=1.4707]

  Early stopping at epoch 62

DONE — best loss: 2.127317 | stopped at epoch: 62


### 7. Training Curves

In [ ]:
ep = range(1, len(history['loss'])+1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(ep, history['loss'],        label='total')
axes[0,0].plot(ep, history['loss_normal'], label='normal stream')
axes[0,0].plot(ep, history['loss_over'],   label='synthetic-overexposed stream')
ax2 = axes[0,0].twinx()
ax2.plot(ep, history['w_normal_over'], label='w_normal_over', color='gray', linestyle='--', alpha=0.6)
ax2.set_ylabel('w_normal_over')
axes[0,0].set_title('Loss per stream (curriculum)'); axes[0,0].set_xlabel('Epoch')
axes[0,0].legend(fontsize=8, loc='upper left'); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(ep, history['spa'],                    label='L_spa')
axes[0,1].plot(ep, np.array(history['exp'])*20,       label='L_exp\u00d720')
axes[0,1].plot(ep, np.array(history['col'])*5,        label='L_col\u00d75')
axes[0,1].plot(ep, np.array(history['ill'])*400,      label='L_ill\u00d7400')
axes[0,1].plot(ep, np.array(history['noise'])*50,     label='L_noise\u00d750')
axes[0,1].set_title('Loss Components (dark stream)')
axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)

axes[1,0].semilogy(ep, history['lr'], color='red', linewidth=2)
axes[1,0].set_title('Learning Rate'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(ep, np.log10(np.array(history['loss'])+1e-8), linewidth=2)
axes[1,1].set_title('log10(Loss) Convergence'); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history_pp.png', dpi=150)
plt.show()
print('Saved: training_history_pp.png')
print(f'Final: loss_normal={history["loss_normal"][-1]:.4f} (lower = better at leaving correct images alone)')
print(f'Final: loss_over={history["loss_over"][-1]:.4f} (lower = better at darkening blown-out images)')

### 8. Inference — trained denoiser + optional bilateral filter

The denoiser is now trained, not just applied at inference. `denoise=True`
still runs an additional bilateral filter pass on top for real-world photos
if you want extra smoothing; set it `False` to evaluate the model's own
learned denoising alone (e.g. for PSNR/SSIM against LOL-v2's Normal set).

In [9]:
def enhance_single_image(image_path, checkpoint=CHECKPOINT,
                          se_positions=(3,), reduction=4,
                          se_activation='relu', se_gate='sigmoid',
                          denoise=True, d=9, sigma_color=75, sigma_space=75):
    ckpt = torch.load(checkpoint, map_location=device)

    m = DCENetLite(use_attention=True, se_positions=se_positions, reduction=reduction,
                   se_activation=se_activation, se_gate=se_gate).to(device)
    m.load_state_dict(ckpt['model'])
    m.eval()

    dn = DenoiseHead().to(device)
    dn.load_state_dict(ckpt['denoise_head'])
    dn.eval()

    img = Image.open(image_path).convert('RGB')
    inp = TF.to_tensor(img).unsqueeze(0).to(device)

    with torch.no_grad():
        enh = dn(enhance_image_with_curve(inp, m(inp)))

    enh_np = (enh.squeeze(0).cpu().permute(1,2,0).numpy()*255).astype(np.uint8)

    if denoise:
        bgr    = cv2.cvtColor(enh_np, cv2.COLOR_RGB2BGR)
        bgr    = cv2.bilateralFilter(bgr, d=d, sigmaColor=sigma_color, sigmaSpace=sigma_space)
        enh_np = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(np.array(img)); axes[0].set_title('Input (low-light)');  axes[0].axis('off')
    axes[1].imshow(enh_np);        axes[1].set_title('Enhanced (Zero-DCE++ + DenoiseHead)' + (' + Bilateral' if denoise else ''))
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('inference_pp.png', dpi=150)
    plt.show()
    return enh_np

print('Usage:')
print('  enhance_single_image(\"your_photo.jpg\", denoise=True)   # real-world photos, extra smoothing on top')
print('  enhance_single_image(\"lolv2_test.jpg\", denoise=False)  # metric evaluation, learned denoiser only')
print('  # match se_positions/reduction/se_activation/se_gate to whatever CHECKPOINT was trained with')

Usage:
  enhance_single_image("your_photo.jpg", denoise=True)   # real-world photos, extra smoothing on top
  enhance_single_image("lolv2_test.jpg", denoise=False)  # metric evaluation, learned denoiser only
  # match se_positions/reduction/se_activation/se_gate to whatever CHECKPOINT was trained with


---
### Version History

| Version | Key issue | Status |
|---|---|---|
| v1-v6 | Various (see earlier notebooks) | Superseded |
| v7 | Depth-separable convs + single shared 3-channel curve (genuine Zero-DCE++). SE-Block fixed after Conv3. | Good baseline |
| v8 | SE-Block placement/activation configurable. Added trained `DenoiseHead` + `NoiseSuppressionLoss`. | **Broken** \u2014 noise loss collapsed training to black by epoch 10 |
| v9 | Noise loss fixed (excess-gradient, warm-up, detached from curve network). `MAX_EXPOSURE` soft-ceiling (post-curve). Adaptive `ExposureControlLoss` target. | PSNR 20.06 / SSIM 0.696 / LPIPS 0.265 pass targets. Real-world photos still overexposed well-lit input / never darkened blown-out input (network never trained on non-dark examples) |
| v10 | Added normal-exposure + synthetic-overexposed training streams, simultaneous, equal weight, from epoch 1 | **Regressed** \u2014 dark-stream `mean_brightness` stuck at 0.17-0.35 (worse than v9's 0.27-0.49, target 0.52); `loss_normal`/`loss_over` never converged either. Three competing objectives fighting an 11K-param model from a random start. |
| **v11 (this notebook)** | Curriculum: dark-only for 60 epochs (matches v9), then normal/over streams ramp in over 40 epochs to a lower target weight (0.3, not 1.0). Early-stop baseline reset when ramp completes, since loss scale isn't comparable before/after. Streams 2/3 skipped entirely (not run at zero weight) before the ramp, so pre-ramp epochs cost the same as v9, not 3x. | **Current** \u2014 compare dark-stream `mean_brightness` trajectory against v9/v10, and run the evaluation notebook's exposure robustness check (Normal images fed directly + synthetic-overexposed) to confirm the curriculum actually fixed what v10 didn't |

**For BAB III/IV:** the v9\u2192v10\u2192v11 progression is a genuinely strong story if your
timeline allows writing it up fully \u2014 you have concrete evidence (training logs) of
a specific failure mode (naive multi-objective training regressing the primary task)
and a specific, principled fix (curriculum staging + reduced target weight), not just
"we tried X and it worked." Report mean_brightness trajectories for v9/v10/v11 side by
side as a figure if you have room \u2014 it's a clearer illustration than the loss curves
alone.
